In [ ]:
%pip install numpy pandas numba duckdb plotly

In [ ]:
import os
import sys

REPO_URL  = "https://github.com/payamdav/pycrypto2.git"
REPO_NAME = "pycrypto2"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

In [ ]:
# ── Parameters (edit here) ──────────────────────────────────
asset     = "btcusdt"
date_from = "2026-06-01"
date_to   = "2026-06-07 23:59:00"   # inclusive, minute resolution
# ────────────────────────────────────────────────────────────

In [ ]:
from packages.candle_loader import load_candles

candles = load_candles(asset, date_from, date_to)
# columns: ts, o, h, l, c, v, q, n, vwap, vb, vs
ts   = candles[:, 0]
o, h, l, c = candles[:, 1], candles[:, 2], candles[:, 3], candles[:, 4]
vwap = candles[:, 8]

print(f"Loaded {len(ts)} candles for {asset} [{date_from} .. {date_to}]")

In [ ]:
import pandas as pd
from packages.indicators.rolling_mean_stddev import rolling_mean_stddev

# ── Window (edit here) ──────────────────────────────────────
window = 60
# ────────────────────────────────────────────────────────────

mean_std = rolling_mean_stddev(vwap, window)
stddev   = mean_std[:, 1]
variance = stddev ** 2

print(pd.Series(stddev, name="rolling_stddev").describe())
print()
print(pd.Series(variance, name="rolling_variance").describe())

In [ ]:
# ── Kalman parameters (edit here) ────────────────────────────
process_variance           = 1e-4   # smoothed (kalman_1d_batch) — Q
measurement_variance       = 1.0    # smoothed (kalman_1d_batch) — R
process_variance_adaptive  = 1e-4   # smoothed_adaptive (kalman_1d_batch_adaptive) — Q
# ────────────────────────────────────────────────────────────

In [ ]:
from packages.kalman_filter import kalman_1d_batch, kalman_1d_batch_adaptive

smoothed, _ = kalman_1d_batch(
    vwap, vwap[0], 1.0, process_variance, measurement_variance
)
smoothed_adaptive, _ = kalman_1d_batch_adaptive(
    vwap, process_variance_adaptive, window
)

In [ ]:
from packages.indicators.rolling_vwap import rolling_vwap

# ── Rolling VWAP windows (edit here) ────────────────────────
vwap_window_fast = 10
vwap_window_slow = 60
# ────────────────────────────────────────────────────────────

v, q = candles[:, 5], candles[:, 6]
rolling_vwap_fast = rolling_vwap(q, v, vwap_window_fast)
rolling_vwap_slow = rolling_vwap(q, v, vwap_window_slow)

In [ ]:
import plotly.graph_objects as go

dt = pd.to_datetime(ts, unit="ms", utc=True)

fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=dt, open=o, high=h, low=l, close=c, name=asset,
))
fig.add_trace(go.Scatter(x=dt, y=vwap, name="vwap",
                          line=dict(color="#f5a623", width=1.5)))
fig.add_trace(go.Scatter(x=dt, y=smoothed, name="smoothed",
                          line=dict(color="#1f77b4", width=2)))
fig.add_trace(go.Scatter(x=dt, y=smoothed_adaptive, name="smoothed_adaptive",
                          line=dict(color="#9467bd", width=2)))
fig.add_trace(go.Scatter(x=dt, y=rolling_vwap_fast, name=f"rolling_vwap_{vwap_window_fast}",
                          line=dict(color="#2ca02c", width=1.5)))
fig.add_trace(go.Scatter(x=dt, y=rolling_vwap_slow, name=f"rolling_vwap_{vwap_window_slow}",
                          line=dict(color="#d62728", width=1.5)))

fig.update_layout(
    template="plotly_white",
    title=f"{asset} — {date_from} to {date_to} — Kalman smoothing vs adaptive",
    hovermode="x unified",
    xaxis_rangeslider_visible=False,
    height=700,
    margin=dict(l=40, r=40, t=60, b=40),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_xaxes(range=[dt.min(), dt.max()])

fig.show()